In [45]:
import os
import rootutils
import pandas as pd
import random
import numpy as np

import matplotlib.pyplot as plt

from tqdm.notebook import tqdm
tqdm.pandas()

rootutils.setup_root(os.path.abspath('./'), indicator=".project-root", pythonpath=True, dotenv=True, cwd=True)

# auto-loading of imports from outside scripts
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [46]:
from src.cif_utils import cif_from_file

In [47]:
meta_csv = "data_cod/cod_bradley_merged.csv"
cifs_dir = "cifs"

In [48]:
meta_df = pd.read_csv(meta_csv).drop_duplicates().sort_values(by="id")
meta_df["cif_path"] = cifs_dir + "/" + meta_df["id"].astype(str) + ".cif"

---
## Coordinational Numbers:

In [49]:
from pymatgen.core import Structure
import numpy as np
import pandas as pd
from collections import defaultdict

from src.cif_utils import get_fixed_length_descriptor 

In [52]:
cutoffs = {
    ("O", "H"): 1.27,
    ("O", "C"): 1.72,
    ("C", "H"): 1.37,
    ("C", "N"): 1.60,
    ("C", "C"): 1.70,
    ("O", "O"): 1.50,
    # ("C", "S"): 1.90,
    # ("O", "O"): 1.50
}

cutoffs = {
    ("C", "H"): 1.20,   # C–H bond length ≈ 1.09 Å + buffer
    ("C", "C"): 1.70,   # C–C single bond ≈ 1.54 Å + buffer
    ("C", "N"): 1.60,   # C–N single bond ≈ 1.47 Å + buffer
    ("C", "O"): 1.60,   # C–O single bond ≈ 1.43 Å + buffer
    ("C", "S"): 1.90,   # C–S single bond ≈ 1.82 Å + buffer
    ("C", "Cl"): 1.90,  # C–Cl bond length ≈ 1.76 Å + buffer
    ("C", "Br"): 2.00,  # C–Br bond length ≈ 1.94 Å + buffer
    ("C", "I"): 2.20,   # C–I bond length ≈ 2.14 Å + buffer
    ("N", "H"): 1.10,   # N–H bond length ≈ 1.01 Å + buffer
    ("N", "O"): 1.50,   # N–O single bond ≈ 1.45 Å + buffer
    ("N", "N"): 1.50,   # N–N single bond ≈ 1.45 Å + buffer
    ("O", "H"): 1.10,   # O–H bond length ≈ 0.97 Å + buffer
    ("O", "O"): 1.50,   # O–O single bond ≈ 1.45 Å + buffer
    ("S", "H"): 1.40,   # S–H bond length ≈ 1.34 Å + buffer
    ("S", "O"): 1.60,   # S–O single bond ≈ 1.48 Å + buffer
    ("Cl", "H"): 1.30,  # H–Cl bond length ≈ 1.27 Å + buffer
    ("Br", "H"): 1.50,  # H–Br bond length ≈ 1.41 Å + buffer
    ("I", "H"): 1.70,   # H–I bond length ≈ 1.61 Å + buffer
}


In [53]:

error_cifs = [] 
for i, cif_path in tqdm(enumerate(meta_df["cif_path"]), total=len(meta_df), desc="Processing CIFs"):
    try:
        coord_numbs = get_fixed_length_descriptor(
            cif_path,
            cutoffs,
            use_interior=True,
            struct_repeat=(2, 2, 2)
        )
    except Exception as e:
        print(f"Error: {e}")
        error_cifs.append(cif_path)
        
    print(coord_numbs)
    print(f"smiles: {meta_df.iloc[i]['can_smiles']}")
    if i >= 10:
        break


Processing CIFs:   0%|          | 0/13528 [00:00<?, ?it/s]

Br–H    0.000000
H–Br    0.000000
C–Br    0.000000
Br–C    0.000000
C–C     2.323529
C–Cl    0.000000
Cl–C    0.000000
C–H     1.347222
H–C     0.875536
C–I     0.000000
I–C     0.000000
C–N     0.000000
N–C     0.000000
C–O     0.279412
O–C     1.000000
C–S     0.000000
S–C     0.000000
Cl–H    0.000000
H–Cl    0.000000
I–H     0.000000
H–I     0.000000
N–H     0.000000
H–N     0.000000
N–N     0.000000
N–O     0.000000
O–N     0.000000
O–H     1.000000
H–O     0.140496
O–O     0.000000
S–H     0.000000
H–S     0.000000
S–O     0.000000
O–S     0.000000
Name: avg_cn_descriptor, dtype: float64
smiles: CC1(C)[C@@H]2CC[C@]3(C2)[C@H](O)CC[C@@H](O)[C@]13O.CC1(C)[C@H]2CC[C@@]3(C2)[C@@H](O)CC[C@H](O)[C@@]13O
Br–H    0.000000
H–Br    0.000000
C–Br    0.000000
Br–C    0.000000
C–C     1.875706
C–Cl    0.000000
Cl–C    0.000000
C–H     0.938776
H–C     0.956989
C–I     0.000000
I–C     0.000000
C–N     0.152542
N–C     3.000000
C–O     0.112994
O–C     1.000000
C–S     0.000000
S–C     0.000000